# HashAtlas walkthrough
Can a small Bitcoin miner still win? This notebook loads the pipeline outputs (run `uv run hashatlas run-all` first) and re-derives headline numbers independently.

In [1]:
import json, pandas as pd, numpy as np
from pathlib import Path
A = Path('../artifacts')
q = json.load(open(A/'quality_report.json'))
print('blocks:', q['rows_loaded'], '| heights', q['height_range'])
print('label audit agreement:', round(q['label_audit']['agreement'], 4), 'on', q['label_audit']['blocks_compared'], 'blocks')

blocks: 975494 | heights [0, 975493]
label audit agreement: 0.9415 on 424464 blocks


## 1. Who mines? Unattributed share and solo blocks by year

In [2]:
b = pd.read_parquet('../data/interim/blocks.parquet', columns=['time','miner','is_solo'])
y = b.groupby(b['time'].dt.year).agg(blocks=('miner','size'), unknown=('miner', lambda s: (s=='Unknown').mean()), solo=('is_solo','sum'))
y.round(3)

,blocks,unknown,solo
time,,,
2009,32490,1.0,0
2010,67920,1.0,0
2011,59627,0.965,0
2012,54526,0.64,0
2013,63433,0.178,0
2014,58865,0.142,12
2015,54321,0.059,141
2016,54851,0.016,81
2017,55928,0.036,21


## 2. Fingerprint classifier: recompute macro-F1 from the saved test predictions

In [3]:
p = pd.read_parquet('../data/processed/test_predictions.parquet')
from sklearn.metrics import f1_score, confusion_matrix
print('macro-F1', round(f1_score(p['y'], p['pred'], average='macro'), 4))
tm = json.load(open(A/'test_metrics.json'))
print('published', round(tm['families'][tm['operational_family']]['macro_f1'], 4))
pd.crosstab(p['y'], p['pred'], normalize='index').round(2)

macro-F1 0.2032
published 0.2438


pred,AntPool,BTC.TOP,Binance Pool,Braiins Pool,CloverPool,F2Pool,Foundry USA,Huobi.pool,Other pools,Poolin,Solo,ViaBTC
y,,,,,,,,,,,,
AntPool,0.74,0.0,0.03,0.00,0.00,0.00,0.16,0.00,0.03,0.00,0.00,0.03
Binance Pool,0.54,0.0,0.07,0.00,0.00,0.00,0.24,0.00,0.04,0.00,0.00,0.10
Braiins Pool,0.01,0.0,0.01,0.07,0.01,0.00,0.28,0.01,0.47,0.00,0.01,0.13
CloverPool,0.42,0.0,0.12,0.00,0.01,0.00,0.27,0.00,0.04,0.00,0.00,0.13
F2Pool,0.03,0.0,0.08,0.00,0.00,0.63,0.09,0.12,0.00,0.00,0.00,0.05
Foundry USA,0.14,0.0,0.03,0.03,0.00,0.00,0.46,0.01,0.19,0.00,0.00,0.15
Other pools,0.31,0.0,0.04,0.05,0.01,0.02,0.20,0.01,0.23,0.01,0.01,0.11
Poolin,0.01,0.0,0.02,0.02,0.00,0.00,0.22,0.00,0.27,0.00,0.00,0.44
Solo,0.00,0.0,0.18,0.09,0.00,0.09,0.12,0.00,0.18,0.00,0.29,0.06


## 3. Eras rediscovered by K-means (weekly rows)

In [4]:
w = pd.read_parquet('../data/processed/weeks.parquet')
pd.crosstab(w['kmeans'], w['era'])

era,ASIC,CPU,FPGA/early ASIC,GPU,Industrial ASIC
kmeans,,,,,
0,0,75,2,46,0
1,94,0,85,0,0
2,80,0,0,0,544


## 4. Association rules (top 10 by lift)

In [5]:
r = json.load(open(A/'rules.json'))
pd.DataFrame(r['rules'])[['antecedent','consequent','support','confidence','lift','confirm_confidence']].head(10)

,antecedent,consequent,support,confidence,lift,confirm_confidence
0,"[fullness=full(>=98%), version=signal bits set]",miner=SBI Crypto,0.014800,0.647080,42.973966,0.803625
1,[version=signal bits set],miner=SBI Crypto,0.015057,0.611729,40.626222,0.845280
2,"[fullness=full(>=98%), version=rolls 5-8 bits,...",miner=Foundry USA,0.012992,0.391598,1.288206,0.361740
3,"[version=rolls 5-8 bits, interval=fast(<60s)]",miner=Foundry USA,0.013431,0.381570,1.255216,0.317704
4,"[version=rolls 5-8 bits, interval=fast(<60s), ...",miner=Foundry USA,0.011523,0.381175,1.253917,0.341554
5,"[version=rolls 5-8 bits, txcount=below-day-med...",miner=Foundry USA,0.018832,0.373540,1.228801,0.335453
6,"[version=rolls 5-8 bits, content=witness-heavy...",miner=Foundry USA,0.014004,0.373451,1.228510,0.344255
7,"[version=rolls 9-12 bits, txcount=below-day-me...",miner=Foundry USA,0.010992,0.370733,1.219568,0.310183
8,"[version=rolls 9-12 bits, shape=consolidating]",miner=Foundry USA,0.013490,0.370134,1.217599,0.318382
9,"[fullness=full(>=98%), version=rolls 9-12 bits...",miner=Foundry USA,0.013075,0.368483,1.212165,0.351680


## 5. Miner advisor: Hong Kong vs Asia (Antminer S21, business tariff)

In [6]:
d = json.load(open(A/'decisions.json'))
a = pd.DataFrame(d['advisor'])
a[(a.device=='Antminer S21') & (a.tariff=='business') & a.asia][['country','price_usd_kwh','pool_income_usd_day','electricity_usd_day','profit_usd_day','recommendation']].sort_values('price_usd_kwh')

,country,price_usd_kwh,pool_income_usd_day,electricity_usd_day,profit_usd_day,recommendation
135,Bhutan,0.019,7.438811,1.596,5.842811,pool mining (profitable)
125,Indonesia,0.069,7.438811,5.796,1.642811,pool mining (profitable)
123,Vietnam,0.078,7.438811,6.552,0.886811,pool mining (profitable)
131,Kazakhstan,0.078,7.438811,6.552,0.886811,pool mining (profitable)
109,China,0.109,7.438811,9.156,-1.717189,do not mine (unprofitable)
113,South Korea,0.119,7.438811,9.996,-2.557189,do not mine (unprofitable)
129,India,0.122,7.438811,10.248,-2.809189,do not mine (unprofitable)
121,Thailand,0.127,7.438811,10.668,-3.229189,do not mine (unprofitable)
119,Malaysia,0.129,7.438811,10.836,-3.397189,do not mine (unprofitable)
127,Philippines,0.158,7.438811,13.272,-5.833189,do not mine (unprofitable)


## 6. Future Lab: halving timing and backtest

In [7]:
f = json.load(open(A/'future.json'))
print(f['halving'])
pd.DataFrame(f['backtest']['rows'])[['origin','horizon_days','inside_80pct_band','median_error_pct','persistence_error_pct']]

{'height': 1050000, 'date_p10': '2028-04-04', 'date_p50': '2028-04-07', 'date_p90': '2028-04-09'}


,origin,horizon_days,inside_80pct_band,median_error_pct,persistence_error_pct
0,2019-05-01,90,True,-7.252545,-31.058727
1,2019-05-01,180,True,-9.085775,-49.353758
2,2019-05-01,360,True,38.877787,-57.049903
3,2019-05-01,540,False,107.597461,-64.600496
4,2020-05-11,90,True,27.807836,-2.900956
5,2020-05-11,180,False,73.228167,0.186701
6,2020-05-11,360,False,110.367329,-30.591190
7,2020-05-11,540,False,296.513663,-23.327016
8,2022-01-01,90,True,1.595647,-14.018038
9,2022-01-01,180,True,5.943543,-19.763750


## 7. Independent verification

In [8]:
v = json.load(open(A/'verification.json'))
print(v['passed'], '/', v['total'])
pd.DataFrame(v['checks'])[['check','passed']]

17 / 17


,check,passed
0,test macro-F1 from saved predictions,True
1,test accuracy from saved predictions,True
2,confusion matrix sums to test blocks,True
3,no label-source / time features in model,True
4,train < validation < test < new (no overlap),True
5,model choice frozen before test evaluation,True
6,root split Gini reduction,True
7,exported tree (web simulator) == fitted tree,True
8,transactions count,True
9,Apriori brute-force check (stage) passed,True
